In [10]:
import pytesseract
from PIL import Image
import cv2
import os

# Set the path to the Tesseract executable
tesseract_path = r'C:/Program Files/Tesseract-OCR/tesseract.exe'
print(f"Tesseract path: {tesseract_path}")
pytesseract.pytesseract.tesseract_cmd = tesseract_path

def correct_image_orientation(image_path):
    # Load the image using OpenCV
    image = cv2.imread(image_path)
    
    # Convert the image to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    try:
        # Detect orientation and script using Tesseract
        osd = pytesseract.image_to_osd(gray)
        angle = int(osd.split("\n")[2].split(": ")[1])
        script = osd.split("\n")[3].split(": ")[1]

        print(f"Detected angle: {angle}")
        print(f"Detected script: {script}")

        # Rotate the image to correct the orientation
        if angle != 0:
            (h, w) = image.shape[:2]
            center = (w // 2, h // 2)
            M = cv2.getRotationMatrix2D(center, -angle, 1.0)
            rotated = cv2.warpAffine(image, M, (w, h))
        else:
            rotated = image

    except pytesseract.TesseractError as e:
        print(f"Tesseract OSD Error: {e}")
        rotated = image

    # Save the corrected image
    corrected_image_path = 'corrected_' + os.path.basename(image_path)
    cv2.imwrite(corrected_image_path, rotated)
    return corrected_image_path

def extract_text(image_path):
    print(f"Image path: {image_path}")
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")
    
    # Correct the image orientation
    corrected_image_path = correct_image_orientation(image_path)
    
    image = Image.open(corrected_image_path)
    print("Image opened successfully")
    
    # Ensure the image has a valid resolution
    image = image.convert('RGB')
    image.save(corrected_image_path, dpi=(300, 300))
    
    # Perform OCR on the image for both English and Nepali
    try:
        text = pytesseract.image_to_string(image, lang='eng+nep')
        print("OCR completed")
    except pytesseract.TesseractError as e:
        print(f"Tesseract OCR Error: {e}")
        text = ""

    return text

# Example usage
image_path = 'X:\demat confirmation.jpg'
text = extract_text(image_path)
print(text)

Tesseract path: C:/Program Files/Tesseract-OCR/tesseract.exe
Image path: X:\demat confirmation.jpg
Detected angle: 0
Detected script: 7.63
Image opened successfully
OCR completed
NABIL INVESTMENT BANKING LTD.
Account Information
As of Mon, Dec 18, 2023 22:48:46

BOID 1301040002374584
Name PRABIN TIWARI
Account Status ACTIVE

BO Sub Status INDIVIDUAL-RESIDENT
Confirmation Wavied Yes
Gender M
Date Of Birth 2000-10-16
*(ate format: yyyy-mm-dd)
Citizenship Number KAVREPALANCHOWK-30-01-76-04290-
2019
+([ssued district-Citizenship number- Issued year in AD)
PAN Number N/A

Father's/Mother's Name

SHIV PRASAD TIWARI/KALPANA TIWARI

Spouse/GrandFather's Name

/DASHARATH TIWARI

Account Open Date

2023-05-17

*(ate format: yyyy-mm-dd)

Contact Number

9869028215, 9869028215, 9869028215

Email prabintiwari44@gmail.com

Address 05,CHAUNRIDEURALI, BAGMATI,
CHAUNRIDEURALI, KATHMANDU,
KATHMANDU, NEPAL

Bank Name Nabil Bank Ltd.-Personal Lending Branch

Account Number

03410017517116




In [12]:
# import cv2
# import numpy as np

# def capture_image_from_webcam():
#     cap = cv2.VideoCapture(0)
#     while True:
#         ret, frame = cap.read()
#         if not ret:
#             print("Failed to capture image")
#             break
#         cv2.imshow("Press SPACE to capture", frame)
#         if cv2.waitKey(1) & 0xFF == ord(' '):
#             image = frame
#             break
#     cap.release()
#     cv2.destroyAllWindows()
#     return image

# def get_contour_corners(contour):
#     # Get a bounding box around the largest contour and use it to get the 4 corner points
#     rect = cv2.minAreaRect(contour)
#     box = cv2.boxPoints(rect)
#     box = np.int0(box)
#     return box

# def order_points(pts):
#     # Sort the points based on their x-coordinates
#     xSorted = pts[np.argsort(pts[:, 0]), :]

#     # Grab the left-most and right-most points from the sorted x-coordinates
#     leftMost = xSorted[:2, :]
#     rightMost = xSorted[2:, :]

#     # Sort the left-most coordinates according to their y-coordinates, then grab the top-left and bottom-left points
#     leftMost = leftMost[np.argsort(leftMost[:, 1]), :]
#     (tl, bl) = leftMost

#     # Sort the right-most coordinates according to their y-coordinates, then grab the top-right and bottom-right points
#     rightMost = rightMost[np.argsort(rightMost[:, 1]), :]
#     (tr, br) = rightMost

#     # Return the coordinates in top-left, top-right, bottom-right, and bottom-left order
#     return np.array([tl, tr, br, bl], dtype="float32")

# def four_point_transform(image, pts):
#     # Obtain a consistent order of the points and unpack them individually
#     rect = order_points(pts)
#     (tl, tr, br, bl) = rect

#     # Compute the width of the new image, which will be the maximum distance between bottom-right and bottom-left
#     # x-coordinates or the top-right and top-left x-coordinates
#     widthA = np.sqrt(((br[0] - bl[0]) ** 2) + ((br[1] - bl[1]) ** 2))
#     widthB = np.sqrt(((tr[0] - tl[0]) ** 2) + ((tr[1] - tl[1]) ** 2))
#     maxWidth = max(int(widthA), int(widthB))

#     # Compute the height of the new image, which will be the maximum distance between the top-right and bottom-right
#     # y-coordinates or the top-left and bottom-left y-coordinates
#     heightA = np.sqrt(((tr[0] - br[0]) ** 2) + ((tr[1] - br[1]) ** 2))
#     heightB = np.sqrt(((tl[0] - bl[0]) ** 2) + ((tl[1] - bl[1]) ** 2))
#     maxHeight = max(int(heightA), int(heightB))

#     # Set up the destination points to obtain a "birds eye view" (i.e., top-down view) of the image, again specifying
#     # points in the top-left, top-right, bottom-right, and bottom-left order
#     dst = np.array([
#         [0, 0],
#         [maxWidth - 1, 0],
#         [maxWidth - 1, maxHeight - 1],
#         [0, maxHeight - 1]], dtype="float32")

#     # Compute the perspective transform matrix and then apply it
#     M = cv2.getPerspectiveTransform(rect, dst)
#     warped = cv2.warpPerspective(image, M, (maxWidth, maxHeight))

#     return warped

# def process_image(image):
#     gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
#     blurred = cv2.GaussianBlur(gray, (5, 5), 0)
#     edged = cv2.Canny(blurred, 75, 200)

#     contours, _ = cv2.findContours(edged, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
#     contours = sorted(contours, key=cv2.contourArea, reverse=True)[:5]

#     for contour in contours:
#         perimeter = cv2.arcLength(contour, True)
#         approx = cv2.approxPolyDP(contour, 0.02 * perimeter, True)
#         if len(approx) == 4:
#             screenCnt = approx
#             break

#     warped = four_point_transform(gray, screenCnt.reshape(4, 2))
#     scanned = cv2.adaptiveThreshold(warped, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)

#     return scanned

# def save_image(image, filename="scanned_image.jpg"):
#     cv2.imwrite(filename, image)

# if __name__ == "__main__":
#     image = capture_image_from_webcam()
#     scanned_image = process_image(image)
#     save_image(scanned_image)
#     cv2.imshow("Scanned Image", scanned_image)
#     cv2.waitKey(0)
#     cv2.destroyAllWindows()


import cv2

def capture_image_from_webcam_with_annotation():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return None

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Define the rectangle for the document area
    rect_x = int(width * 0.2)
    rect_y = int(height * 0.2)
    rect_w = int(width * 0.6)
    rect_h = int(height * 0.6)

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to capture image")
            break
        
        # Flip the frame horizontally to avoid mirror effect
        frame = cv2.flip(frame, 1)
        
        # Draw the rectangle on the frame
        cv2.rectangle(frame, (rect_x, rect_y), (rect_x + rect_w, rect_y + rect_h), (0, 255, 0), 2)
        cv2.imshow("Align document inside the rectangle and press SPACE to capture", frame)

        if cv2.waitKey(1) & 0xFF == ord(' '):
            # Crop the image within the rectangle
            cropped_image = frame[rect_y:rect_y + rect_h, rect_x:rect_x + rect_w]
            break

    cap.release()
    cv2.destroyAllWindows()
    return cropped_image

def save_image(image, filename="captured_image.jpg"):
    cv2.imwrite(filename, image)
    print(f"Image saved as {filename}")

if __name__ == "__main__":
    image = capture_image_from_webcam_with_annotation()
    if image is not None:
        save_image(image)
    else:
        print("No image captured")

Image saved as captured_image.jpg


In [15]:
#flip the image and after clicking the image it will again flip and stores the image in the directoty

import cv2

def capture_image_from_webcam_with_annotation():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return None

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Define the rectangle for the document area
    rect_x = int(width * 0.2)
    rect_y = int(height * 0.2)
    rect_w = int(width * 0.6)
    rect_h = int(height * 0.6)

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to capture image")
            break
        
        # Flip the frame horizontally to avoid mirror effect
        frame = cv2.flip(frame, 1)
        
        # Draw the rectangle on the frame
        cv2.rectangle(frame, (rect_x, rect_y), (rect_x + rect_w, rect_y + rect_h), (0, 255, 0), 2)
        cv2.imshow("Align document inside the rectangle and press SPACE to capture", frame)

        if cv2.waitKey(1) & 0xFF == ord(' '):
            # Crop the image within the rectangle
            cropped_image = frame[rect_y:rect_y + rect_h, rect_x:rect_x + rect_w]
            
            # Flip the cropped image back to correct the text orientation
            cropped_image = cv2.flip(cropped_image, 1)

            break

    cap.release()
    cv2.destroyAllWindows()
    return cropped_image

def save_image(image, filename="captured_image.jpg"):
    cv2.imwrite(filename, image)
    print(f"Image saved as {filename}")

if __name__ == "__main__":
    image = capture_image_from_webcam_with_annotation()
    if image is not None:
        save_image(image)
    else:
        print("No image captured")

Image saved as captured_image.jpg


In [3]:
import cv2

def capture_image_from_webcam_with_annotation():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return None

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Define the rectangle for the document area
    rect_x = int(width * 0.2)
    rect_y = int(height * 0.2)
    rect_w = int(width * 0.6)
    rect_h = int(height * 0.6)

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to capture image")
            break

        # Flip the frame horizontally to avoid mirror effect
        frame = cv2.flip(frame, 1)

        # Draw the rectangle on the frame
        cv2.rectangle(frame, (rect_x, rect_y), (rect_x + rect_w, rect_y + rect_h), (0, 255, 0), 2)
        cv2.imshow("Align document inside the rectangle and press SPACE to capture", frame)

        if cv2.waitKey(1) & 0xFF == ord(' '):
            # Crop the image within the rectangle
            cropped_image = frame[rect_y:rect_y + rect_h, rect_x:rect_x + rect_w]
            break

    cap.release()
    cv2.destroyAllWindows()
    return cropped_image

def save_image(image, filename="captured_image.jpg"):
    cv2.imwrite(filename, image)
    print(f"Image saved as {filename}")

if __name__ == "__main__":
    image = capture_image_from_webcam_with_annotation()
    if image is not None:
        save_image(image)
    else:
        print("No image captured")


Image saved as captured_image.jpg


In [1]:
import cv2

def capture_image_from_webcam_with_annotation():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return None

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Define the rectangle for the document area
    rect_x = int(width * 0.2)
    rect_y = int(height * 0.2)
    rect_w = int(width * 0.6)
    rect_h = int(height * 0.6)

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to capture image")
            break

        # Flip the frame horizontally to avoid mirror effect
        frame = cv2.flip(frame, 1)

        # Draw the rectangle on the frame
        cv2.rectangle(frame, (rect_x, rect_y), (rect_x + rect_w, rect_y + rect_h), (0, 255, 0), 2)
        cv2.imshow("Align document inside the rectangle and press SPACE to capture", frame)

        if cv2.waitKey(1) & 0xFF == ord(' '):
            # Check the dimensions and coordinates before cropping
            print(f"Cropping image at: x={rect_x}, y={rect_y}, width={rect_w}, height={rect_h}")
            print(f"Frame dimensions: width={frame.shape[1]}, height={frame.shape[0]}")

            # Ensure the coordinates are within bounds
            if rect_x + rect_w <= frame.shape[1] and rect_y + rect_h <= frame.shape[0]:
                cropped_image = frame[rect_y:rect_y + rect_h, rect_x:rect_x + rect_w]
            else:
                print("Error: Cropping coordinates are out of bounds.")
                cropped_image = None
            break

    cap.release()
    cv2.destroyAllWindows()
    return cropped_image

def save_image(image, filename="captured_image.jpg"):
    if image is not None:
        cv2.imwrite(filename, image)
        print(f"Image saved as {filename}")
    else:
        print("No image to save")

if __name__ == "__main__":
    image = capture_image_from_webcam_with_annotation()
    if image is not None:
        save_image(image)
    else:
        print("No image captured")


Cropping image at: x=128, y=96, width=384, height=288
Frame dimensions: width=640, height=480
Image saved as captured_image.jpg


In [11]:
import cv2

def capture_image_from_webcam_with_annotation():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return None

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Define the rectangle for the document area
    rect_x = int(width * 0.2)
    rect_y = int(height * 0.2)
    rect_w = int(width * 0.6)
    rect_h = int(height * 0.6)

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to capture image")
            break
        
        # Flip the frame horizontally to avoid mirror effect
        frame = cv2.flip(frame, 1)
        
        # Draw the rectangle on the frame
        cv2.rectangle(frame, (rect_x, rect_y), (rect_x + rect_w, rect_y + rect_h), (0, 255, 0), 2)
        cv2.imshow("Align document inside the rectangle and press SPACE to capture", frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord(' '):
            # Crop the image within the rectangle
            cropped_image = frame[rect_y:rect_y + rect_h, rect_x:rect_x + rect_w]
            # Flip the cropped image back to correct the text orientation
            cropped_image = cv2.flip(cropped_image, 1)
            break
        elif key == 27:  # Esc key to exit without capturing
            print("Exiting without capturing image")
            cropped_image = None
            break

    cap.release()
    cv2.destroyAllWindows()
    return cropped_image

def save_image(image, filename="captured_image.jpg"):
    if image is not None:
        cv2.imwrite(filename, image)
        print(f"Image saved as {filename}")
    else:
        print("No image to save")

if __name__ == "__main__":
    image = capture_image_from_webcam_with_annotation()
    if image is not None:
        save_image(image)
    else:
        print("No image captured")


Image saved as captured_image.jpg
